<a href="https://colab.research.google.com/github/MartinaTrigilia/warehouse-monitoring-system/blob/main/warehouse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Sistema di monitoraggio ordini e magazzino in Python
Descrizione del progetto
L'azienda LogiServe S.r.l. è una piccola realtà di e-commerce B2B specializzata nella distribuzione di componenti elettrici per officine e rivenditori. Il progetto mira a sviluppare un applicativo a riga di comando o con interfaccia testuale per la gestione del magazzino e degli ordini. L'obiettivo è simulare una soluzione gestionale reale che consenta di registrare nuovi ordini, aggiornare le giacenze dei prodotti, consultare lo stato del magazzino in tempo reale e generare report giornalieri utili per le decisioni operative.

Il progetto pone particolare attenzione a chiarezza, modularità e manutenibilità del codice, e sarà consegnato come file Python contenente la descrizione del progetto, i dati di esempio, e le istruzioni per l'uso.

LogiServe riceve quotidianamente ordini da clienti ricorrenti e nuovi rivenditori. Per evitare rotture di stock, gestire le priorità di evasione e fornire report giornalieri al responsabile di magazzino, l'azienda necessita di uno strumento semplice ma affidabile che:

registri gli ordini in arrivo;
aggiorni automaticamente le giacenze dopo l'evasione;
permetta di verificare disponibilità e soglie di riordino;
produca report giornalieri con sintetiche informazioni su vendite, giacenze e ordini in sospeso.
L'importanza del progetto per LogiServe è elevata: riduce il rischio di out-of-stock, migliora i tempi di evasione ordini e offre dati aggregati utili per decisioni sul riassortimento e sul servizio al cliente.

Obiettivi del progetto
Il progetto dovrà fornire, a livello di requisiti funzionali e non funzionali, le seguenti caratteristiche principali:

Funzionalità principali (requisiti funzionali) 1. Registrazione di nuovi ordini - Inserimento di ordini contenenti: identificativo cliente, lista di prodotti (codice, quantità richiesta), data e priorità. - Verifica dello stato di disponibilità al momento della registrazione (disponibile/parziale/indisponibile).

Aggiornamento delle giacenze

Riduzione delle giacenze al momento di evasione ordine.
Gestione di scorte negative o ordini parziali come stato separato (es. “parziale”, “in attesa”).
Consultazione dello stato del magazzino in tempo reale

Visualizzazione elenco prodotti con giacenza attuale, punto di riordino e unità di misura.
Filtri di ricerca per codice prodotto, categoria o soglia di giacenza.
Generazione di report giornalieri

Report sintetico giornaliero contenente: numero ordini ricevuti, ordini evasi, ordini in sospeso, prodotti più venduti, variazione giacenze.
Esportazione del report in formato leggibile (ad es. file di testo o CSV) per condivisione interna.
Persistenza dei dati (requisiti di alto livello)

Possibilità di caricare e salvare lo stato di magazzino e la lista ordini su file di dati (formato standard come CSV/JSON), in modo che il sistema possa essere riavviato mantenendo lo stato.
Tracciabilità e log

Tracciamento delle operazioni principali (es. registrazione ordine, evasione, modifica giacenze) con timestamp e operatore.
Requisiti non funzionali
Chiarezza e documentazione: codice con spiegazioni chiare delle funzioni, dell’architettura logica e delle istruzioni per l’uso.
Modularità: componenti separate per gestione ordini, magazzino, reportistica e persistenza dati.
Manutenibilità: codice leggibile con nomi significativi e commenti che facilitino estensioni future.
Usabilità: interfaccia testuale intuitiva con menu e messaggi di feedback chiari.
Robustezza: comportamento prevedibile in presenza di input incompleti o non validi (gestione di errori a livello di flusso).
Dati e formato dei file
Dati di esempio da includere nel file Python:

Catalogo prodotti: file con colonne come CodiceProdotto, Nome, Categoria, GiacenzaIniziale, PuntoRiordino, UnitàMisura.
Lista ordini: file con colonne come IDOrdine, Data, Cliente, Stato (nuovo/evaso/parziale), ElencoProdotti (codice+quantità).
Storico operazioni: file log delle operazioni (timestamp, operazione, risorsa, dettaglio).
I file dovrebbero essere in formati standard (ad es. CSV o JSON) per permettere il caricamento e il salvataggio dello stato. Alla prima esecuzione del programma, se i file non esistono già, il programma deve crearli con dati iniziali simulati per gli scopi di questo esercizio.

Casi di test e scenari di valutazione
Esempi di scenari utili per valutare il progetto:

Scenari di ordine evaso completamente: verifica giacenze aggiornate correttamente.
Ordine parziale: verifica dello stato dell’ordine e della giacenza residua.
Scarico consistente: verifica del comportamento al raggiungimento del punto di riordino o giacenza zero.
Import/export dei dati: caricamento dello stato iniziale e salvataggio dello stato finale.
Valore aggiunto per l'azienda
Riduzione del rischio di rottura di stock grazie a una visibilità immediata delle giacenze e alle notifiche dei punti di riordino.
Migliore capacità di pianificazione degli approvvigionamenti: report giornalieri sintetici per il responsabile acquisti.
Migliore servizio clienti: informazioni tempestive sulla disponibilità dei prodotti e sui tempi di evasione.
Base riutilizzabile e manutenibile per futuri sviluppi (integrazione con sistemi esterni, interfacce grafiche, moduli di analisi storica).


In [ ]:
import csv, string, random, logging
from datetime import datetime, date

In [ ]:
class Product:
  """
  This class rapresents a product of the warehouse
  """
  def __init__(self,product_code,product_name,product_cat,product_stock,reorder_point,product_udm):

    """
    product_code (string): alfanumeric string that identifies a product

    product_name (string): name or description of a product

    product_cat (string): category of a product

    product_stock (int) : product stock

    product_udm (int) : indicates the unit of measurement of a product

    """
    self._product_code = product_code
    self._product_name = product_name
    self._product_cat  = product_cat
    self._product_stock = product_stock
    self._reorder_point = reorder_point
    self._product_udm   = product_udm


  @property
  def product_code(self):
      return self._product_code

  @product_code.setter
  def product_code(self, value):
    if not (isinstance(value,str) and value.strip() ):
      raise ValueError("Product code must be a non-empty alphanumeric string")
    self._product_code = value

  @property
  def product_name(self):
      return self._product_name

  @product_name.setter
  def product_name(self, value):
    if not (value.strip() ):
      raise ValueError("Product name must be a non-empty string")
    self._product_name = value

  @property
  def product_cat(self):
      return self._product_cat

  @product_cat.setter
  def product_cat(self, value):
    if not (value.strip() ):
      raise ValueError("Product Category must be a non-empty string")
    self._product_cat = value

  @property
  def product_stock(self):
      return self._product_stock

  @product_stock.setter
  def product_stock(self, value):
      if not (isinstance(value,int) and value >= 0):
          raise ValueError("Product stock must be a positive integer!")
      self._product_stock = value

  @property
  def reorder_point(self):
      return self._reorder_point

  @reorder_point.setter
  def reorder_point(self, value):
      if not (isinstance(value,int) and value > 0):
          raise ValueError("Reorder must be a positive integer!")
      self._reorder_point = value

  @property
  def product_udm(self):
      return self._product_udm

  @product_udm.setter
  def product_udm(self, value):
    if not (value.strip() ):
      raise ValueError("Product UDM must be a non-empty string")
    self._product_udm = value

  #  True if a Product is below its reorder point
  #  False otherwise

  def isBelowReorderPt(self):
    if  self._product_stock < self._reorder_point:
      return True
    return False

  @staticmethod
  def get_product_code():
    return 'PRD' + ''.join(random.choices(string.digits, k=5))

  # Logging Method


  def __repr__(self):
      return (f"Product(product_code={self._product_code!r}, product_name={self._product_name!r}, "
              f"product_cat={self._product_cat!r}, product_stock={self._product_stock!r}, "
              f"reorder_point={self._reorder_point!r}, product_udm={self._product_udm!r}))")

  def __str__(self):
      return (f"Product( \nProduct Code: {self._product_code}, \nProduct Name: {self._product_name}, "
              f"\nProduct Category: {self._product_cat}, \nProduct Stock: {self._product_stock}, "
              f"\nProduct Reorder Point: {self._reorder_point} , \nProduct Udm: {self._product_udm})")


class Order:
  """
  This class rapresents an order made by a client
  """
  def __init__(self,order_id,order_dt,client_id,state,priority,product_list):

    """
    order_id (string): alfanumeric string that identifies an order

    order_dt (date): datetime of an order

    client_id (string): alfanumeric string that identifies the client that has made an order

    state (int) : integer that rapresents the state of an order. 1 <= state <= 3

    Order State Workflow:

    1 -- > Order Recorded
    2 -- > Order Processed
    3 -- > Order Suspended

    priority (int) : integer that rapresents the state of an order. 1 <= priority <= 4

    Order Fulfillment Priority:

    1 --> Low
    2 --> Standard
    3 --> High
    4 --> Very High

    product_list (list) : a list where each item is a tuple (product_id,quantity)

    """
    self._order_id = order_id
    self._order_dt = order_dt
    self._client_id  = client_id
    self._state = state
    self._priority = priority
    self._product_list = product_list


  @property
  def order_id(self):
      return self._order_id

  @order_id.setter
  def order_id(self, value):
    if not (isinstance(value,str) and value.strip() ):
      raise ValueError("Order id must be a non-empty alphanumeric string")
    self._order_id = value

  @property
  def order_dt(self):
      return self._order_dt

  @order_dt.setter
  def order_dt(self, value):
      if not (isinstance(value,date)):
          raise ValueError("Order Date must be a valid date")
      self._order_dt = value

  @property
  def client_id(self):
      return self._client_id

  @client_id.setter
  def client_id(self, value):
    if not (isinstance(value,str) and value.strip() ):
      raise ValueError("Client id must be a non-empty alphanumeric string")
    self._client_id = value

  @property
  def state(self):
      return self._state

  @state.setter
  def state(self, value):
      if not ( value >= 1 and value <=3 and isinstance(value,int)):
        raise ValueError("Order state not valid!")
      self._state = value

  @property
  def priority(self):
      return self._priority

  @priority.setter
  def priority(self, value):
      if not ( value >= 1 and value <=4 and isinstance(value,int)):
          raise ValueError("Order priority not valid!")
      self._priority = value

  @property
  def product_list(self):
      return self._product_list

  @product_list.setter
  def product_list(self, value):
    if not isinstance(value, list):
        raise ValueError("Product list must be a list")
    for item in value:
        if not (isinstance(item, tuple) and len(item) == 2):
            raise ValueError("Each product must be a tuple (product_id, quantity)")
        product_id, quantity = item
        if not (isinstance(product_id, str) and product_id.strip()):
            raise ValueError("Product ID must be a non-empty string")
        if not (isinstance(quantity, int) and quantity > 0):
            raise ValueError("Quantity must be a positive integer")
    self._product_list = value

  @staticmethod
  def get_order_code():
    return 'ORD' + ''.join(random.choices(string.digits, k=5))

  # Logging Method

  def __repr__(self):
      return (f"Order(order_id={self._order_id!r}, order_dt={self._order_dt!r}, "
              f"client_id={self._client_id!r}, state={self._state!r}, "
              f"priority={self._priority!r}, product_list={self._product_list!r}))")

  def __str__(self):
      return (f"Order( \nOrder Id: {self._order_id}, \nOrder Date: {self._order_dt}, "
              f"\nClient Id: {self._client_id}, \nOrder State: {self._state}, "
              f"\nOrder Priority: {self._priority})")

class Client:
  """
  This class rapresents a client
  """
  def __init__(self,client_id,full_name,address,postal_code,vat_number):

    """
    client_id (string): alfanumeric string that identifies a client id

    full_name (string): name and surname of a client

    address (string): shipping address of a client

    postal_code (int) : postal code of the address of a client

    vat_number (string) : alfanumeric string that identifies the vat_number of a client


    """
    self._client_id = client_id
    self._full_name = full_name
    self._address  = address
    self._postal_code = postal_code
    self._vat_number = vat_number



  @property
  def client_id(self):
      return self._client_id

  @client_id.setter
  def client_id(self, value):
    if not (isinstance(value,str) and value.strip() ):
      raise ValueError("Client ID must be a non-empty alphanumeric string")
    self._client_id = value

  @property
  def full_name(self):
      return self._full_name

  @full_name.setter
  def full_name(self, value):
    if not (value.strip() ):
      raise ValueError("Client name must be a non-empty string")
    self._full_name = value

  @property
  def address(self):
      return self._address

  @address.setter
  def address(self, value):
    if not (isinstance(value,str) and value.strip() ):
      raise ValueError("Client Address must be a non-empty string")
    self._address = value

  @property
  def postal_code(self):
      return self._postal_code

  @postal_code.setter
  def postal_code(self, value):
    if not (isinstance(value,int) and value > 0):
      raise ValueError("Client Post Code must be a non-negative integer")
    self._postal_code = value

  @property
  def vat_number(self):
      return self._vat_number

  @vat_number.setter
  def vat_number(self, value):
    if not (isinstance(value,str) and value.strip() ):
      raise ValueError("Client Vat Number must be a non-empty alphanumeric string")
    self._vat_number = value

  # Logging Method

  def __repr__(self):
      return (f"Client(client_id={self.client_id!r}, full_name={self.full_name!r}, "
              f"address={self.address!r}, postal_code={self.postal_code!r}, "
              f"vat_number={self.vat_number!r})")

  def __str__(self):
      return (f"Client( \nClient Id: {self.client_id}, \nFull Name: {self.full_name}, "
              f"\nClient Address: {self.address}, \nClient Postal Code: {self.postal_code}, "
              f"\nClient Vat Number: {self.vat_number})")

class Warehouse:

    # class attributes
    product_category = {
      'Resistor': 'Passive Components',
      'Capacitor': 'Passive Components',
      'Inductor': 'Passive Components',
      'Diode': 'Passive Components',
      'Transformer': 'Passive Components',
      'Transistor': 'Active Components',
      'Integrated Circuit': 'Active Components',
      'Voltage Regulator': 'Active Components',
      'LED': 'Active Components',
      'Relay': 'Electromechanical Components',
      'Switch': 'Electromechanical Components',
      'Connector': 'Electromechanical Components',
      'Fuse': 'Electromechanical Components',
      'Battery': 'Power Supplies',
      'Power Adapter': 'Power Supplies',
      'Cable': 'Cables & Wiring',
      'Terminal': 'Cables & Wiring',
      'Temperature Sensor': 'Sensors',
      'Light Sensor': 'Sensors',
  }

    units_mapping = {
        'Passive Components': 'pcs',
        'Active Components': 'pcs',
        'Electromechanical Components': 'pcs',
        'Power Supplies': 'pcs',
        'Cables & Wiring': 'm',
        'Sensors': 'pcs',
    }

  #    client_db = {
  #     'Resistor': 'Passive Components',
  #     'Capacitor': 'Passive Components',
  #     'Inductor': 'Passive Components',
  #     'Diode': 'Passive Components',
  #     'Transformer': 'Passive Components',
  #     'Transistor': 'Active Components',
  #     'Integrated Circuit': 'Active Components',
  #     'Voltage Regulator': 'Active Components',
  #     'LED': 'Active Components',
  #     'Relay': 'Electromechanical Components',
  #     'Switch': 'Electromechanical Components',
  #     'Connector': 'Electromechanical Components',
  #     'Fuse': 'Electromechanical Components',
  #     'Battery': 'Power Supplies',
  #     'Power Adapter': 'Power Supplies',
  #     'Cable': 'Cables & Wiring',
  #     'Terminal': 'Cables & Wiring',
  #     'Temperature Sensor': 'Sensors',
  #     'Light Sensor': 'Sensors',
  # }

    def __init__(self, product_list=None, order_list=None,client_list=None):

        self._product_list = {} if product_list is None else {p.product_code: p for p in product_list}
        self._order_list = {} if order_list is None else {o.order_id: o for o in order_list}
        self._client_list = {} if client_list is None else {o.client_id: c for c in client_list}


    # Method to manage products in the warehouse

    def add_product(self, product):
      if product._product_code in self._product_list:
        raise ValueError("Product already exists!")
      self._product_list[product._product_code] = product

    def delete_product(self, product_code):
      if product_code not in self._product_list:
        raise ValueError("Product doesn't exist!")
      self._product_list.pop(product_code)

    def update_stock(self, product_code, qta):
      if product_code not in self._product_list:
        raise ValueError("Product doesn't exist!")
      self._product_list[product_code]._product_stock = qta

    def get_product_code_unique(self):
      while True:
        product_code = Product.get_product_code()
        if product_code not in self._product_list:
          return product_code

    # Method to manage orders in the warehouse

    def add_order(self, order):
      if order._order_id in self._order_list:
          raise ValueError("Order already exists!")
      self._order_list[order.order_id] = order

    def update_order(self, order):
      if order._order_id not in self._order_list:
        raise ValueError("Order doesn't exists!")
      self._order_list[order.order_id] = order

    def get_order_code_unique(self):
      while True:
        order_code = Order.get_order_code()
        if order_code not in self._order_list:
          return order_code


In [ ]:
import csv, string, random, logging
from google.colab import files
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
  )

def prc_log(message, level="INFO"):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"{level} [{timestamp}] {message}")

def main():

  warehouse = Warehouse(list(),list())

## Creation of a product catalogue csv file
  filename = 'product_catalogue.csv'
  fieldnames = ['product_code', 'product_name', 'product_cat', 'product_stock', 'reorder_point', 'product_udm']
  leng = 8

  with open(filename, mode='w', newline='') as csvfile:
      writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
      writer.writeheader()

      for _ in range(20):
          product_name = random.choice(list(warehouse.product_category.keys()))
          product_cat = warehouse.product_category[product_name]
          product_code = warehouse.get_product_code_unique()
          product_stock = random.randint(30, 100)
          reorder_point = random.randint(5, 20)
          product_udm = warehouse.units_mapping.get(product_cat, 'pcs')

          writer.writerow({
              'product_code': product_code,
              'product_name': product_name,
              'product_cat': product_cat,
              'product_stock': product_stock,
              'reorder_point': reorder_point,
              'product_udm': product_udm
          })

  prc_log(f"Product catalogue file '{filename}' successfully created with 20 entries.")

  logging.info(f"Product catalogue file '{filename}' successfully created with 20 entries.")

  files.download(filename)

## Creation of a order list csv file
  filename = 'order_list' + datetime.now().strftime('%d%m%Y_%H%M%S') + '.csv'
  fieldnames = ['order_id', 'order_dt', 'client_id', 'order_state', 'product','qta']

  with open(filename, mode='w', newline='') as csvfile:
      writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
      writer.writeheader()

      for _ in range(20):

          order_id = warehouse.get_order_code_unique()
          order_dt = datetime.now().strftime('%d%m%Y_%H%M%S')
          client_id = random.choices(string.ascii_letters + string.digits, k=leng)
          qta = random.randint(5, 20)

          writer.writerow({
              'order_id': order_id,
              'order_dt': order_dt,
              'client_id': client_id,
              'order_state': 1, #recorded
              'qta': qta,
          })

  prc_log(f"Product catalogue file '{filename}' successfully created with 20 entries.")

  logging.info(f"Product catalogue file '{filename}' successfully created with 20 entries.")
  files.download(filename)


main()

INFO [2026-07-11 15:24:50] Product catalogue file 'product_catalogue.csv' successfully created with 20 entries.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

INFO [2026-07-11 15:24:50] Product catalogue file 'order_list11072026_152450.csv' successfully created with 20 entries.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

print(os.listdir('/content'))

import pandas as pd

df = pd.read_csv('product_catalogue.csv')
print(df.head())

['.config', 'order_list11072026_144231.csv', 'order_list11072026_152450.csv', 'order_list11072026_151858.csv', 'order_list11072026_144200.csv', 'product_catalogue.csv', 'sample_data']
  product_code  product_name                   product_cat  product_stock  \
0     PRD43619     Connector  Electromechanical Components             88   
1     PRD17370           LED             Active Components             84   
2     PRD22857      Inductor            Passive Components             35   
3     PRD12062  Light Sensor                       Sensors             98   
4     PRD79555      Inductor            Passive Components             93   

   reorder_point product_udm  
0             10         pcs  
1              8         pcs  
2             20         pcs  
3              8         pcs  
4             19         pcs  


In [ ]:
# Test Product
p = Product("PRD00001", "Resistor", "Passive Components", 50, 10, "pcs")
print(p)
print(repr(p))
print(f"Below reorder? {p.isBelowReorderPt()}")

# Test Order
o = Order("ORD00001", date.today(), "CLI001", 1, 2, [("PRD00001", 5)])
print(o)
print(repr(o))

# Test Client
c = Client("CLI001", "Mario Rossi", "Via Roma 1", 20100, "IT12345678901")
print(c)
print(repr(c))

Product( 
Product Code: PRD00001, 
Product Name: Resistor, 
Product Category: Passive Components, 
Product Stock: 50, 
Product Reorder Point: 10 , 
Product Udm: pcs)
Product(product_code='PRD00001', product_name='Resistor', product_cat='Passive Components', product_stock=50, reorder_point=10, product_udm='pcs'))
Below reorder? False
Order( 
Order Id: ORD00001, 
Order Date: 2026-07-11, 
Client Id: CLI001, 
Order State: 1, 
Order Priority: 2)
Order(order_id='ORD00001', order_dt=datetime.date(2026, 7, 11), client_id='CLI001', state=1, priority=2, product_list=[('PRD00001', 5)]))
Client( 
Client Id: CLI001, 
Full Name: Mario Rossi, 
Client Address: Via Roma 1, 
Client Postal Code: 20100, 
Client Vat Number: IT12345678901)
Client(client_id='CLI001', full_name='Mario Rossi', address='Via Roma 1', postal_code=20100, vat_number='IT12345678901')
